# BigQuery Direct Ingestion & Chronological Split Strategies

This notebook demonstrates how to build and execute Vertex AI AutoML Tabular pipelines directly ingesting data from **BigQuery tables** (`bq://project.dataset.table`) and configuring **predefined time-based or custom data splitting strategies** (`predefined_split_key`).

## Direct BigQuery Ingestion vs Cloud Storage CSV
- **Direct Ingestion**: Referencing BigQuery tables via `bq://project.dataset.table` avoids extra GCS export steps, preserves strict schema typings, and leverages BigQuery native querying performance.
- **Predefined Split Strategies**: Using `predefined_split_key` allows data scientists to control dataset partition assignments (`TRAIN`, `VALIDATE`, `TEST`, or `UNASSIGNED`) directly within a BigQuery column (e.g. for time-series / chronological evaluation preventing future-data leakage).

## Workflow Highlights
1. **Pipeline Configuration**: Define `TabularPipelineConfig` with `bigquery_table_path` and `predefined_split_key`.
2. **Stage 1 PipelineJob**: Compile and inspect Full Architecture Search pipeline job reading from BigQuery.
3. **Stage 2 Skip Architecture Search PipelineJob**: Compile and inspect Stage 2 pipeline job referencing the BigQuery dataset.
4. **Vertex AI Experiment Tracking**: Automatic logging and inspection of pipeline runs in Vertex AI Experiments.

In [ ]:
from dotenv import load_dotenv

from tabflows import (
    TabularPipelineConfig,
    create_tabular_pipeline_job,
    list_experiment_runs,
    run_skip_architecture_search_pipeline,
)

# Load environment variables from local .env file
load_dotenv()
print("Environment setup and tabflows imports completed.")

Environment setup and tabflows imports completed.


In [ ]:
# Initialize TabularPipelineConfig with BigQuery direct ingestion
# and predefined time-based split key
config = TabularPipelineConfig(
    bigquery_table_path="bq://hybrid-vertex.bank_dataset.train",
    predefined_split_key="split_col",
    experiment_name="automl-tabular-classification-experiments",
)

print(f"Project ID: {config.project_id}")
print(f"BigQuery Table Path: {config.bigquery_table_path}")
print(f"Predefined Split Key: {config.predefined_split_key}")
print(f"Experiment Name: {config.experiment_name}")

Project ID: hybrid-vertex
BigQuery Table Path: bq://hybrid-vertex.bank_dataset.train
Predefined Split Key: split_col
Experiment Name: automl-tabular-classification-experiments


In [ ]:
# Create Stage 1 AutoML Tabular PipelineJob reading directly from BigQuery table
stage_1_job = create_tabular_pipeline_job(
    config=config,
    job_id="automl-tabular-bq-stage-1-full-search",
    log_experiment=True,
)

print(f"Stage 1 BigQuery PipelineJob created and tracked in experiment '{config.experiment_name}'.")
print(f"Pipeline Root Directory: {config.root_dir}")

Stage 1 BigQuery PipelineJob created and tracked in experiment 'automl-tabular-classification-experiments'.
Pipeline Root Directory: gs://jts-tabflows-v1/automl_tabular_pipeline


In [ ]:
from tabflows import get_task_detail

# Extract tuning_result_output artifact URI or fallback to pipeline root location
try:
    has_gca = getattr(stage_1_job, "_gca_resource", None) is not None
    if has_gca and stage_1_job.gca_resource is not None:
        pipeline_task_details = stage_1_job.gca_resource.job_detail.task_details
        stage_1_tuner_task = get_task_detail(pipeline_task_details, "automl-tabular-stage-1-tuner")
        if stage_1_tuner_task:
            tuning_result_uri = stage_1_tuner_task.outputs["tuning_result_output"].artifacts[0].uri
        else:
            tuning_result_uri = f"{config.root_dir}/tuning_result_output_artifact"
    else:
        tuning_result_uri = f"{config.root_dir}/tuning_result_output_artifact"
except Exception:
    tuning_result_uri = f"{config.root_dir}/tuning_result_output_artifact"

# Configure Stage 2 Skip Architecture Search reusing BigQuery source and tuning results
stage_2_config = TabularPipelineConfig(
    bigquery_table_path="bq://hybrid-vertex.bank_dataset.train",
    predefined_split_key="split_col",
    run_architecture_search=False,
    tuning_result_output=tuning_result_uri,
    experiment_name="automl-tabular-classification-experiments",
)

# Create Stage 2 Skip Architecture Search pipeline job referencing BigQuery source
stage_2_job = run_skip_architecture_search_pipeline(
    config=stage_2_config,
    tuning_result_artifact_uri=tuning_result_uri,
    job_id="automl-tabular-bq-stage-2-skip-search",
    log_experiment=True,
)

print(
    f"Stage 2 BigQuery PipelineJob created and tracked in experiment "
    f"'{stage_2_config.experiment_name}'."
)

Stage 2 BigQuery PipelineJob created and tracked in experiment 'automl-tabular-classification-experiments'.


In [ ]:
# Retrieve and display experiment runs logged in Vertex AI Experiments
print(f"Fetching logged experiment runs for '{config.experiment_name}'...")
try:
    df_runs = list_experiment_runs(config=config)
    if df_runs is not None and len(df_runs) > 0:
        print(f"Found {len(df_runs)} experiment run(s):")
        print(df_runs)
    else:
        print(f"No experiment runs found under '{config.experiment_name}'.")
except Exception as e:
    print(f"Notice: Vertex AI Experiment '{config.experiment_name}' query status: {e}")

Fetching logged experiment runs for 'automl-tabular-classification-experiments'...
No experiment runs found under 'automl-tabular-classification-experiments'.
